# 00 — Data Verification

Verifies KITTI and CityPersons split integrity before any training begins.

**What this notebook does:**
1. Asserts zero overlap between train/val/test for both datasets
2. Prints a summary table: image count per split + occlusion level distribution
3. Plots occlusion level distributions per split (saved to `results/figures/`)

**Run this once** before starting notebook 01.

In [ ]:
import sys
sys.path.insert(0, '..')  # allow imports from project root

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Project root
ROOT = Path('..').resolve()
DATA_ROOT = ROOT / 'data'
FIGURES_DIR = ROOT / 'results' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', ROOT)
print('Data root:   ', DATA_ROOT)

## 1. KITTI Split Verification

In [ ]:
import importlib.util, sys as _sys
spec = importlib.util.spec_from_file_location(
    'split_verification', ROOT / 'data' / 'split_verification.py'
)
sv = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sv)

kitti_root = DATA_ROOT / 'kitti'
kitti_splits = sv.load_kitti_splits(kitti_root)

if any(kitti_splits.values()):
    sv.assert_no_overlap(kitti_splits, 'KITTI')
    kitti_occ = sv.load_kitti_occlusion_distribution(kitti_root, kitti_splits)
    sv.print_summary_table('KITTI Pedestrian', kitti_splits, kitti_occ)
    total = sum(len(v) for v in kitti_splits.values())
    print(f'Total KITTI images accounted for: {total} (expected ≈7481)')
else:
    print('[SKIP] KITTI data not found at', kitti_root)

## 2. CityPersons Split Verification

In [ ]:
cp_root = DATA_ROOT / 'citypersons'
cp_splits = sv.load_citypersons_splits(cp_root)

if any(cp_splits.values()):
    sv.assert_no_overlap(cp_splits, 'CityPersons')
    cp_occ = sv.load_citypersons_occlusion_distribution(cp_root, cp_splits)
    sv.print_summary_table('CityPersons Pedestrian', cp_splits, cp_occ)
    total = sum(len(v) for v in cp_splits.values())
    print(f'Total CityPersons images accounted for: {total} (expected ≈5050)')
else:
    print('[SKIP] CityPersons data not found at', cp_root)

## 3. Occlusion Level Distribution Plots

In [ ]:
def plot_occlusion_distribution(occ_dist, dataset_name, save_path):
    """Bar chart of occlusion level distribution per split."""
    splits = [s for s in ['train', 'val', 'test'] if s in occ_dist]
    levels = [0, 1, 2, 3]
    level_labels = ['lvl 0\n(fully visible)', 'lvl 1\n(partly)', 'lvl 2\n(largely)', 'lvl 3\n(unknown)']
    colours = ['#2196F3', '#FF9800', '#F44336', '#9E9E9E']

    fig, axes = plt.subplots(1, len(splits), figsize=(5 * len(splits), 4), sharey=False)
    if len(splits) == 1:
        axes = [axes]

    for ax, split in zip(axes, splits):
        counts = [occ_dist[split].get(lvl, 0) for lvl in levels]
        bars = ax.bar(level_labels, counts, color=colours, edgecolor='white', linewidth=0.5)
        ax.set_title(f'{split.capitalize()} split')
        ax.set_ylabel('Annotation count')
        ax.set_xlabel('Occlusion level')
        for bar, count in zip(bars, counts):
            if count > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                        str(count), ha='center', va='bottom', fontsize=9)

    fig.suptitle(f'{dataset_name} — Occlusion Level Distribution per Split', fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.savefig(str(save_path).replace('.png', '.pdf'), bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')


if any(kitti_splits.values()):
    plot_occlusion_distribution(
        kitti_occ, 'KITTI Pedestrian',
        FIGURES_DIR / 'kitti_occlusion_distribution.png'
    )

if any(cp_splits.values()):
    plot_occlusion_distribution(
        cp_occ, 'CityPersons Pedestrian',
        FIGURES_DIR / 'citypersons_occlusion_distribution.png'
    )

## 4. Combined figure — saved to results/figures/

In [ ]:
# Save a combined split_occlusion_distribution.png with both datasets side by side
# Only runs if both datasets are present
if any(kitti_splits.values()) and any(cp_splits.values()):
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    levels = [0, 1, 2, 3]
    level_labels = ['lvl0', 'lvl1', 'lvl2', 'lvl3']
    colours = ['#2196F3', '#FF9800', '#F44336', '#9E9E9E']

    for row, (occ_dist, name) in enumerate([(kitti_occ, 'KITTI'), (cp_occ, 'CityPersons')]):
        for col, split in enumerate(['train', 'val', 'test']):
            ax = axes[row][col]
            counts = [occ_dist.get(split, {}).get(lvl, 0) for lvl in levels]
            ax.bar(level_labels, counts, color=colours, edgecolor='white')
            ax.set_title(f'{name} — {split}', fontsize=10)
            ax.set_ylabel('Annotations')

    plt.suptitle('Occlusion Level Distributions: KITTI and CityPersons', fontsize=13)
    plt.tight_layout()
    out = FIGURES_DIR / 'split_occlusion_distribution.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.savefig(str(out).replace('.png', '.pdf'), bbox_inches='tight')
    plt.show()
    print(f'Combined figure saved: {out}')
else:
    print('[INFO] Combined figure requires both datasets. Skipping.')